# P03 — Viviendas turísticas: Moran global e incremental

## Objetivos
1. Delimitar población, campos y soporte antes del análisis.
2. Verificar calidad y construir ICOUNT conservando coordenadas y registros.
3. Interpretar Moran global y dos exploraciones de escala con mapas/gráficos ArcGIS.
4. Separar magnitud, significancia y causa.

**Pregunta:** ¿los conteos de viviendas registradas en sitios vecinos se parecen más de lo esperado bajo el modelo nulo, y cómo depende de la escala?

**Procedencia:** Instituto Distrital de Turismo, Subdirección de Inteligencia y Gestión de Tecnologías de la Información; distribuido por Catastro/IDECA. Catálogo https://www.ideca.gov.co/recursos/mapas/vivienda-turistica ; contenido indicado 31/05/2025, consulta/copia 2026-09-15. Posiciones aproximadas de oferta inferior a 30 días; no todo el mercado ni datos comerciales Airbnb. Uso bajo términos del productor, sin presumir licencia CC.

**Copia mínima:** 7 529 registros del endpoint autorizado https://serviciosgis.catastrobogota.gov.co/arcgis/rest/services/turismo/turismobogota/MapServer/22 ; solo OBJECTID, LOCALIDAD, SUBCATRNT y geometría. No se consultaron nombres/direcciones. Conteo inicial/final iguales; IDs únicos/completos; lotes de 1 000 sin exceso de transferencia. JSON y Procedencia.md acompañan la GDB. Se consume offline, sin redescargar; la instantánea puede diferir de junio.

**Requisitos:** ArcGIS Pro 3.6, licencia disponible, kernel limpio; ejecutar en orden. Sin ModelBuilder, otro notebook ni auxiliar compartido. Cambie solo DATA_DIR/OUTPUT_DIR para rutas externas.

| Etapa | Por qué | Entrada → salida | Interpretación |
| --- | --- | --- | --- |
| EDA | Delimitar registro | Campos/XY/CRS → perfiles/mapa/barras | Nulos y coincidencias no son limpieza |
| Copia/CheckGeometry | Proteger original | Registros → copia comprobada | No certifica geocodificación |
| Collect Events | Crear atributo | XY exacta → ICOUNT | Sitios ocupados, no tasas |
| Moran global | Contrastar asociación | ICOUNT → índice/z/p/HTML | Inversa, fila, umbral automático |
| Incremental inicial | Explorar escala | ICOUNT → 20 bandas/curvas | Inicio 2 000, paso 200 m |
| Incremental segunda | Refinar exploración | ICOUNT → 30 bandas/curvas | Inicio 1 000, paso 200 m |
| Síntesis | Acotar evidencia | Tablas/mapas → conclusión | No validación independiente |

- Distinguir distribución descriptiva de multiplicidad y agrupación local Gi* con ocho vecinos/FDR, complemento solicitado.

**Ejecución incompleta:** 2/26 celdas intentadas; 2.832 s; retorno **1**. ArcPy informó `The Product License has not been initialized`. No se ejecutaron modelos ni exportaciones de esta revisión. Las cifras históricas posteriores NO constituyen resultados de esta corrida.

**Presentación aún en revisión:** faltan ajustes y comprobaciones visuales; esto no es entrega académica final. La licencia bloqueó la continuación de las dos últimas prácticas. La captura sin InteractiveShell evitó el error fatal GIL en tres procesos, pero no certifica un kernel interactivo sano.

### Convención visual y conexión
Fondo claro; títulos Cambria y texto Segoe UI en mapas. Colores de grupos nominales, sin orden de riesgo. La base **Light Gray Canvas**, seleccionada del catálogo ArcGIS, aporta nombres y vías de contexto de Esri y colaboradores; la atribución está dentro del marco. **Solo la base requiere conexión**: análisis, datos y PNG incrustados son locales. No modifica coordenadas ni pesos. Si el contexto no carga, el mapa no queda verificado.

En curvas z de Moran, 0 y ±1.96 son referencias **nominales de una prueba fijada**, no corrección FDR ni confirmación tras explorar bandas. Consulte todos los valores en la tabla.

## EDA: registro turístico y soporte observado
La copia mínima representa filas del registro turístico con posiciones aproximadas. No mide disponibilidad actual, visitantes, precio, ocupación ni toda la oferta informal. La fecha del catálogo no es una observación temporal de cada vivienda.

| Campo | Significado/unidad | Decisión |
| --- | --- | --- |
| LOCALIDAD | Categoría administrativa declarada | Frecuencias/barras; no promedio de códigos |
| SUBCATRNT | Subcategoría nominal del registro turístico | Frecuencias/nulos, no escala ordinal inventada |
| OBJECTID/OID | Identidad local de fila | Unicidad operativa, no vivienda única certificada |
| XY / CRS 3857 | Posición aproximada, metros cartográficos | Coincidencias/finitud y distorsión |
| ICOUNT | Registros turísticos por sitio ocupado | Multiplicidad, no ocupación hotelera |

Las barras por localidad cuentan registros, no superficie ni intensidad por habitante. Su desigualdad combina fenómeno y cobertura. Sin campo fecha no hay tendencias ni estacionalidad; 31/05/2025 pertenece al catálogo. Compartir coordenadas puede reflejar edificios o precisión de localización: no autoriza deduplicar.

### Configuración única
Cambie DATA_DIR solo por otra entrada autorizada; OUTPUT_DIR recibe resultados aislados. Esta celda define rutas, no crea datos ni calcula modelos.

**Jupyter / VS Code:** seleccione el intérprete Python de ArcGIS Pro con licencia para ejecutar. Python normal sin `arcpy` no indica fallo del algoritmo. Los PNG incrustados se leen sin ejecutar ni instalar un kernel. No se certifica aquí la interfaz gráfica de VS Code.

### Configuración y estado de esta revisión
La primera celda solo define rutas. `ROOT = None` busca ambos marcadores desde la carpeta actual o sus ancestros. Si Jupyter inicia fuera del vault, sustituya **esa primera asignación** por `ROOT = Path(...)` con la ruta absoluta de su copia; no use `__file__`. `DATA_DIR` y `OUTPUT_DIR` admiten rutas externas absolutas: cambie sus asignaciones aquí y mantenga entradas y salidas separadas. En Bomberos P02, `PREPARED_JURIS` configura la entrada preparada por separado. La plantilla se obtiene de la instalación de Pro. El preflight muestra nombre, ruta resuelta y existencia antes de crear salidas; no instala ArcPy ni obtiene licencia automáticamente.

**Estado N02:** salidas y cifras incrustadas son **históricas, pendiente N04** (ejecución final completa). Las pruebas de rutas no ejecutan modelos. Pendientes N03 (visualización) y N05 (verificación independiente). Se conservan registros previos: cero errores de celda no demuestra cierre natural; el cierre forzado del driver histórico de Clase 02 tampoco lo acredita.


In [1]:
# ROOT=None busca el vault; fuera de él escriba aquí Path con su ruta absoluta.
from pathlib import Path
ROOT = None
ROOT = Path(ROOT).expanduser().resolve() if ROOT is not None else next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'99 - Recursos/notebooks').is_dir()), None)
if ROOT is None or not (ROOT/'AGENTS.md').is_file() or not (ROOT/'99 - Recursos/notebooks').is_dir():
    raise FileNotFoundError('No se encontró el vault (AGENTS.md + 99 - Recursos/notebooks). Edite ROOT = Path con la ruta absoluta del vault; no use __file__.')
DATA_DIR=ROOT/'99 - Recursos/datos/clase_02_viviendas_turisticas/viviendas.gdb'
OUTPUT_DIR=ROOT/'99 - Recursos/salidas_clase_02/practica_03'


### Preflight de solo lectura
Comprobar kernel, licencia y requisitos antes de crear salidas. Cada ausencia se informa por su nombre y ruta exactos.

In [2]:
# Solo lectura: comprobar kernel, licencia y cada ruta antes de crear salidas.
try:
    import arcpy
except Exception as exc:
    raise RuntimeError('ArcPy no disponible. Seleccione el kernel Python de ArcGIS Pro con licencia y reinícielo; no se instala automáticamente.') from exc
try:
    INFO_PRO = arcpy.GetInstallInfo()
    LICENCIA = arcpy.ProductInfo()
except Exception as exc:
    raise RuntimeError('No se pudo inicializar Pro. Revise instalación y acceso a licencia antes de modelar.') from exc
print('ArcGIS Pro:', INFO_PRO.get('Version'), '| licencia:', LICENCIA)
if LICENCIA not in ('ArcView', 'ArcEditor', 'ArcInfo'):
    raise RuntimeError('Licencia no inicializada: ' + str(LICENCIA))
PLANTILLA = Path(INFO_PRO['InstallDir']) / 'Resources/ArcToolBox/Services/routingservices/data/Blank.aprx'
source = str((DATA_DIR / 'Viviendas_turisticas_Bogota').resolve())
template = PLANTILLA
REQUERIDOS = [('Viviendas_turisticas_Bogota', source, False), ('Blank.aprx instalada', template, True)]
faltantes = []
for nombre, ruta, es_archivo in REQUERIDOS:
    ruta = Path(ruta).resolve()
    existe = ruta.is_file() if es_archivo else bool(arcpy.Exists(str(ruta)))
    print(f'{nombre} | {ruta} | existe={existe}')
    if not existe:
        faltantes.append(f'{nombre}: {ruta}')
if faltantes:
    raise FileNotFoundError('Entradas ausentes; ajuste la configuración sin sustituir datos: ' + '; '.join(faltantes))


RuntimeError: ArcPy no disponible. Seleccione el kernel Python de ArcGIS Pro con licencia y reinícielo; no se instala automáticamente.

**Lectura:** `existe=True` indica disponibilidad, no calidad geométrica ni resultados. **Compruebe:** ¿usa el kernel de Pro y la entrada autorizada?

**Resultado:** ubicaciones configuradas; que no haya imagen es esperado. La entrada, los campos y el CRS se comprueban antes de escribir resultados.

## 1. Entorno y campos
ArcPy calcula y dibuja; Counter resume sin editar; math comprueba finitud; csv conserva agregados; uuid aísla corridas; time mide cómputo; IPython incrusta salidas. No se instala nada.

LOCALIDAD y SUBCATRNT son categorías, no magnitudes para promediar. OID identifica filas locales, no viviendas universales; los IDs de consulta se conservan en el JSON mínimo. No se consultó REGNTURISM porque la pregunta no lo necesita. No hay fecha individual en la copia: 31/05/2025 pertenece al catálogo.

Fuente 102100/latest3857, copia 3857. Metros cartográficos no eliminan distorsión Web Mercator. No asignar otro EPSG ni proyectar silenciosamente. La entrada es de solo lectura.

In [ ]:
# Fallar de forma explícita por entrada ausente antes de crear carpetas o salidas GP.
import arcpy, math, csv, uuid, time
from collections import Counter
from IPython.display import display, Image
desc = arcpy.Describe(source)
sr = desc.spatialReference
if desc.shapeType != 'Point' or sr.factoryCode != 3857:
    raise ValueError('Se requiere revisar CRS y unidades de puntos; no asignar/proyectar automáticamente.')
fields = arcpy.ListFields(source)
print('Entorno:', arcpy.GetInstallInfo()['Version'], arcpy.ProductInfo())
print('CRS:', sr.name, sr.factoryCode, sr.linearUnitName)
print('OID identifica fila, no vivienda certificada; la clave de registro real debe verificarse en la fuente.')

if not {'LOCALIDAD','SUBCATRNT'}.issubset({f.name for f in fields}): raise ValueError('Faltan categorías de la copia mínima.')


**Lectura del esquema mínimo:** LOCALIDAD/SUBCATRNT permiten resumir categorías, no precios/demanda. El OID identifica filas, no certifica viviendas únicas.

### Diccionario: leer campos, no promediar identificadores
**Entrada:** esquema real. **Operación:** identificar rol/unidad y límites documentales sin inferir semántica del nombre. **Salida:** tabla visible con ejemplos solo de categorías no personales. Campos sin significado verificado no se analizan.

In [ ]:
# Diccionario visible: esquema real, sin exponer identificadores ni datos personales.
from IPython.display import HTML
from html import escape

def tabla_visible(headers, rows):
    # HTML básico compatible con Jupyter/VS Code; todas las filas son visibles.
    head = ''.join('<th>' + escape(str(v)) + '</th>' for v in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(v)) + '</td>' for v in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

roles = {
    'IncidentesBomberos_SERVICIO': ('Categoría', 'Etiqueta registrada de servicio; no severidad', 'sin unidad'),
    'IncidentesBomberos_CLASE_DE_S': ('Selección temática', 'Etiqueta para seleccionar abejas; nulo no significa ausencia', 'sin unidad'),
    'IncidentesBomberos_NUMERO_INC': ('Clave candidata', 'Identificador registrado; unicidad semántica no certificada', 'sin unidad'),
    'IncidentesBomberos_FECHA': ('Cobertura temporal', 'Fecha registrada; significado operativo exacto no documentado', 'fecha'),
    'Tipo': ('Categoría', 'Etiqueta de tipo; catálogo semántico no certificado', 'sin unidad'),
    'Ciudad': ('Categoría', 'Etiqueta territorial; no límite administrativo', 'sin unidad'),
    'LOCALIDAD': ('Categoría', 'Etiqueta de localidad de la copia; no polígono', 'sin unidad'),
    'SUBCATRNT': ('Categoría', 'Subcategoría registrada; alcance del catálogo no certificado', 'sin unidad'),
}
dictionary_rows = []
for field in arcpy.ListFields(source):
    role, meaning, unit = roles.get(field.name, ('No analizado', 'Semántica no verificada: no inferir del nombre', 'no verificada'))
    example = 'No se expone: innecesario para la pregunta'
    if field.type == 'OID':
        role, meaning, unit = 'Clave local', 'Identifica fila de esta copia, no entidad real única', 'sin unidad'
    elif field.type == 'Geometry':
        role, meaning, unit = 'Posición', 'Geometría puntual; coincidencia no prueba duplicación', sr.linearUnitName
        example = 'Point (tipo geométrico observado)'
    elif role in ('Categoría', 'Selección temática'):
        with arcpy.da.SearchCursor(source, [field.name]) as cursor:
            values = Counter(row[0] for row in cursor)
        example = next((str(v) for v in values if v is not None and str(v).strip()), 'sin ejemplo no nulo')
    elif role == 'Cobertura temporal':
        example = 'Consultar intervalo agregado en la celda siguiente'
    dictionary_rows.append([field.name, field.type, role, meaning, unit, example,
                            'NULL = ausente, no cero; significado administrativo no certificado',
                            '0 no significa ausencia sin diccionario fuente'])
tabla_visible(['Campo real', 'Tipo', 'Rol', 'Significado y límite', 'Unidad', 'Ejemplo seguro', 'Nulo', 'Cero'], dictionary_rows)


**Lectura:** el tipo técnico no certifica significado administrativo. Las claves se revisan por unicidad, nunca por promedio; NULL, vacío y cero se cuentan separadamente a continuación.
**Compruebe:** ¿un número justifica promediar una clave, o cero sustituir un ausente?


### Inspección de calidad sin limpieza
El cursor lee campos pertinentes. Contamos nulos/ceros/categorías, claves y coincidencias; las XY deben ser finitas. Extensión y fechas delimitan cobertura, no precisión ni exhaustividad.

In [ ]:
# Resumir coordenadas y tiempo sin imprimir nombres, direcciones ni registros individuales.
xy_before = Counter()
keys = Counter()
with arcpy.da.SearchCursor(source, ['OID@', 'SHAPE@XY']) as cursor:
    for oid, xy in cursor:
        keys[oid] += 1
        if xy is None or any(v is None or not math.isfinite(v) for v in xy):
            raise ValueError('XY no válida: detener sin reparar ni descartar registros.')
        xy_before[xy] += 1
n = sum(xy_before.values())
print('Filas:', n, 'XY únicos:', len(xy_before), 'coincidencias adicionales:', n-len(xy_before))
print('OID nulos:', keys[None], 'OID cero:', keys[0], 'OID repetidos:', sum(v>1 for v in keys.values()))
if not xy_before or n < 30:
    raise ValueError('Entrada vacía o inesperadamente pequeña; verificar integridad antes del análisis.')
date_fields = [f.name for f in fields if f.type in ('Date', 'DateOnly', 'TimestampOffset')]
if not date_fields:
    print('Sin campo temporal: no atribuir fecha de actualización ni período de oferta.')
for field in date_fields:
    with arcpy.da.SearchCursor(source, [field]) as cursor:
        dates = [row[0] for row in cursor]
    valid = [value for value in dates if value is not None]
    print(field, 'nulos:', len(dates)-len(valid), 'rango:', (min(valid), max(valid)) if valid else None, '| significado temporal pendiente del diccionario')

# Categorías necesarias, sin nombres ni direcciones individuales.
category_counts={f:Counter() for f in ('LOCALIDAD','SUBCATRNT')}
with arcpy.da.SearchCursor(source,['LOCALIDAD','SUBCATRNT']) as cursor:
    for local,sub in cursor:
        category_counts['LOCALIDAD'][local]+=1;category_counts['SUBCATRNT'][sub]+=1
for field,values in category_counts.items():
    print(field,'nulos',values[None],'vacíos',sum(v for k,v in values.items() if isinstance(k,str) and not k.strip()),'ceros',values[0],'frecuencias',dict(values))

# Extensión y ceros diagnostican cobertura sin corregir posiciones automáticamente.
print('Extensión (Xmin,Ymin,Xmax,Ymax), metros del CRS:',
      (min(x[0] for x in xy_before), min(x[1] for x in xy_before),
       max(x[0] for x in xy_before), max(x[1] for x in xy_before)))
print('Registros X=0 o Y=0:', sum(count for (x,y),count in xy_before.items() if x == 0 or y == 0))
print('Sitios coincidentes:', sum(count > 1 for count in xy_before.values()), '| no equivale a duplicados')


**Lectura:** posiciones repetidas no prueban duplicados; pueden corresponder a varias unidades registradas o al mismo edificio. Se mantienen todos los registros. La inspección de OID es operativa; la clave temática no se inventa y debe verificarse cuando se disponga de la fuente. El período de un campo no demuestra cobertura completa. Ningún resultado permite extrapolar a toda la oferta turística.

## 2. EDA visual y espacio de trabajo
El mapa presenta ubicaciones; una barra de multiplicidad XY muestra cuántos sitios contienen uno, dos o más registros, antes de geoprocesar Collect Events. La barra es un agregado de inspección, no un reemplazo del resultado nativo. Todas las salidas y scratch quedan en una carpeta nueva; no se guarda la plantilla ni se descargan mapas base.

**Esta ejecución:** 7 529 filas en 2 919 XY únicos, 624 sitios coincidentes y 4 610 registros adicionales en sitios ya ocupados. No se detectaron OID repetidos/nulos/ceros ni XY no finitas/cero. LOCALIDAD contiene 19 códigos y SUBCATRNT cuatro; los más frecuentes son LOCALIDAD «2» (2 454) y SUBCATRNT «17» (6 297). Se conservan como etiquetas: sin diccionario verificado no se inventan sus nombres ni se promedian. Categorías sin nulos/vacíos observados no prueban cobertura completa; sin fecha individual no hay serie temporal.
**Compruebe:** ¿un número justifica promediar una clave, o cero sustituir un ausente?


In [ ]:
# Crear salidas solo después de superar las verificaciones de entrada y procedencia.
run = OUTPUT_DIR.resolve() / ('ejecucion_' + uuid.uuid4().hex[:12])
run.mkdir(parents=True, exist_ok=False)
arcpy.env.overwriteOutput = False
arcpy.env.addOutputsToMap = False
gdb = str(run / 'trabajo.gdb')
arcpy.management.CreateFileGDB(str(run), 'trabajo.gdb')
arcpy.env.workspace = gdb
arcpy.env.scratchWorkspace = gdb
arcpy.env.outputCoordinateSystem = sr
started = time.perf_counter()
print('Ejecución nueva:', run.name)


**Resultado:** el nombre identifica la corrida actual; informes y capas quedan separados del registro original, sin guardar la plantilla instalada.

### Definir visuales del registro turístico
Funciones locales crean mapa con CRS/leyenda y gráficos ArcGIS. El APRX conserva la copia para inspección manual; la extensión no certifica límites ni cobertura.

In [ ]:
# Solo presentación: no cambia modelos, datos ni coordenadas.
def base_contextual(project, mapa):
    disponibles = project.listBasemaps()
    elegida = next((v for v in ('Light Gray Canvas', 'Human Geography Map') if v in disponibles), None)
    if elegida is None:
        raise RuntimeError('No hay base neutra disponible; el contexto requiere conexión.')
    mapa.addBasemap(elegida)
    print('Base en línea:', elegida, '| Esri y colaboradores; atribución dentro del mapa.')
    return elegida

def texto_atlas(project, layout, geometry, text_type, text, text_size, font_family_name, font_style_name):
    # Ajustar líneas, no reducir pies hasta volverlos ilegibles.
    import textwrap
    ancho = 32 if geometry.X >= 9 else (135 if text_size <= 10 else 80)
    texto = chr(10).join(textwrap.fill(line, width=ancho) for line in text.splitlines())
    return project.createTextElement(layout, geometry, text_type, texto, text_size,
                                     'Cambria' if text_size >= 14 else 'Segoe UI', font_style_name)

def estilo_chart(chart, name):
    # Valor de tema probado mediante exportación real en Pro 3.6.2.
    chart.theme = 'Light'
    chart.displaySize = [1100, 680]
    if name.endswith('_z_score'):
        for valor, etiqueta in [(-1.96, '−1.96 nominal'), (0, 'Referencia nula'), (1.96, '+1.96 nominal')]:
            chart.yAxis.addGuide(arcpy.charts.Guide('line', valueFrom=valor, label=etiqueta,
                                lineColor='#b36b32', lineWidth=1, lineDashStyle='dash'))
        chart.yAxis.minimum = -2.5
    return chart

# Código local de visualización ArcGIS; no depende de auxiliares personalizados externos.
def rectangle(x0, y0, x1, y1):
    return arcpy.Polygon(arcpy.Array([arcpy.Point(x0,y0), arcpy.Point(x1,y0), arcpy.Point(x1,y1), arcpy.Point(x0,y1)]))

def map_png(fc, name, weighted=False):
    project = arcpy.mp.ArcGISProject(str(template))
    m = project.createMap(name, 'MAP')
    for base in m.listLayers():
        m.removeLayer(base)  # Solo capas del mapa nuevo en memoria; no borrar datos.
    base_contextual(project, m)
    m.spatialReference = sr
    layer = m.addDataFromPath(fc)
    layer.name = 'ICOUNT por sitio' if weighted else 'Viviendas registradas'
    sym = layer.symbology
    if weighted:
        sym.updateRenderer('GraduatedSymbolsRenderer')
        sym.renderer.classificationField = 'ICOUNT'
    else:
        sym.renderer.symbol.color = {'RGB': [15, 118, 110, 65]}
        sym.renderer.symbol.size = 2.5
    layer.symbology = sym
    layout = project.createLayout(12, 9, 'INCH', name)
    frame = layout.createMapFrame(rectangle(.3,1,8.8,8), m, 'Mapa')
    frame.camera.setExtent(desc.extent)
    frame.camera.scale *= 1.08
    legend = layout.createMapSurroundElement(rectangle(9,1.5,11.8,7.8), 'LEGEND', frame, None, 'Leyenda')
    legend.fittingStrategy = 'AdjustFontSize'
    texto_atlas(project, layout, arcpy.Point(.4,8.5), 'POINT', 'Viviendas turísticas registradas: ' + ('ICOUNT' if weighted else 'ubicaciones'), 14, 'Arial', 'Regular')
    footer = f'N arriba | {sr.name} | escala 1:{round(frame.camera.scale):,}\nCatastro/IDECA: registros, no toda la oferta. Base en línea: Light Gray Canvas; conteos no son tasas.'
    texto_atlas(project, layout, arcpy.Point(.4,.6), 'POINT', footer, 8, 'Arial', 'Regular')
    path = run / (name + '.png')
    layout.exportToPNG(str(path), resolution=160)
    project.saveACopy(str(run / (name + '.aprx')))
    display(Image(width=1100, filename=str(path)))

def chart_png(chart, name):
    # Conservar y mostrar el mismo PNG nativo, sin sustitución por Matplotlib.
    path = run / (name + '.png')
    estilo_chart(chart, name).exportToPNG(str(path), 1800, 1100)
    display(Image(width=1100, filename=str(path)))


# Tablas y mapas nativos: únicamente atributos de salidas nuevas.
def tabla_frecuencia(nombre, filas):
    table = str(Path(gdb) / nombre)
    arcpy.management.CreateTable(gdb, nombre)
    arcpy.management.AddField(table, 'Categoria', 'TEXT', field_length=100)
    arcpy.management.AddField(table, 'Sitios', 'LONG')
    with arcpy.da.InsertCursor(table, ['Categoria', 'Sitios']) as cursor:
        for row in filas:
            cursor.insertRow(row)
    return table

def mapa_categorias(fc, field, name, title, styles, footer):
    # El pie pertenece a la variable representada, no se reutiliza el pie de OPTICS.
    project = arcpy.mp.ArcGISProject(str(template))
    mapa = project.createMap(name, 'MAP')
    for base in mapa.listLayers():
        mapa.removeLayer(base)
    base_contextual(project, mapa)
    mapa.spatialReference = sr
    layer = mapa.addDataFromPath(fc)
    layer.name = 'ICOUNT' if field == 'ClaseConteo' else 'Gi* con FDR'
    sym = layer.symbology
    sym.updateRenderer('UniqueValueRenderer')
    sym.renderer.fields = [field]
    for group in sym.renderer.groups:
        for item in group.items:
            label, color, size = styles[str(item.values[0][0])]
            item.label = label
            item.symbol.color = {'RGB': color}
            item.symbol.size = size
            item.symbol.outlineColor = {'RGB': [255,255,255,0]}
    layer.symbology = sym
    layout = project.createLayout(12, 9, 'INCH', name)
    def rect(x0,y0,x1,y1):
        return arcpy.Polygon(arcpy.Array([arcpy.Point(x0,y0), arcpy.Point(x1,y0), arcpy.Point(x1,y1), arcpy.Point(x0,y1)]))
    frame = layout.createMapFrame(rect(.3,1.2,8.7,8), mapa, 'Mapa')
    frame.camera.setExtent(desc.extent)
    frame.camera.scale *= 1.08
    legend = layout.createMapSurroundElement(rect(9,2,11.8,7.8), 'LEGEND', frame, None, 'Leyenda')
    legend.fittingStrategy = 'AdjustFontSize'
    texto_atlas(project, layout, arcpy.Point(9,1.8), 'POINT', 'Clases sin puntos: consultar tabla completa.', 9, 'Arial', 'Regular')
    texto_atlas(project, layout, arcpy.Point(.4,8.5), 'POINT', title, 14, 'Arial', 'Regular')
    note = f'N arriba | WKID {sr.factoryCode} | escala 1:{round(frame.camera.scale):,} | base: Light Gray Canvas' + chr(10) + footer
    texto_atlas(project, layout, arcpy.Point(.4,.65), 'POINT', note, 8, 'Arial', 'Regular')
    png = run / (name + '.png')
    layout.exportToPNG(str(png), resolution=160)
    project.saveACopy(str(run / (name + '.aprx')))
    display(Image(width=1100, filename=str(png)))


**Resultado de las definiciones:** visuales preparados, aún no exportados. La cartografía no certifica la posición precisa de la oferta.

### Copia y puerta geométrica

CopyFeatures preserva registros y CheckGeometry detecta problemas antes de modelar. Cero problemas no certifica precisión de posiciones aproximadas.

In [ ]:
# Comprobar la copia sin modificar originales ni reparar automáticamente.
# Inspección gráfica de multiplicidad sin cambiar los registros de origen.
working = str(Path(gdb) / 'Viviendas')
arcpy.management.CopyFeatures(source, working)
geometry_table=str(Path(gdb)/'CheckGeometry_Viviendas')
geometry_result=arcpy.management.CheckGeometry(working,geometry_table)
print(geometry_result.getMessages())
geometry_errors=int(arcpy.management.GetCount(geometry_table)[0])
print('Problemas geométricos:',geometry_errors)
if geometry_errors: raise ValueError('Geometría inválida: detener sin reparar ni eliminar.')


**Lectura y conclusión:** Se informa número de problemas; cualquier error detiene sin reparación ni eliminación. Copiar no es limpiar.

### Mapa, multiplicidad y localidades

Mapa = posiciones; barras de coincidencia = sitios por cantidad de registros; barras de localidad = registros por categoría. Son denominadores distintos.

In [ ]:
# Representar población actual: color, tamaño y frecuencia tienen denominadores distintos.
map_png(working, 'eda_viviendas')
frequency = Counter(xy_before.values())
csv_path = run / 'multiplicidad_xy.csv'
with csv_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.writer(stream); writer.writerow(['Registros_por_XY', 'Sitios'])
    writer.writerows(sorted(frequency.items()))

chart_png(arcpy.charts.Bar(x='LOCALIDAD',aggregation='COUNT',dataSource=working,title='Registros por localidad',rotated=True),'eda_localidad')
# La distribución legible se muestra después de CollectEvents.


**Lectura y conclusión:** La superposición puede ocultar registros; un hueco no demuestra ausencia de turismo ni cobertura completa. Categorías son etiquetas, no valores continuos.

**Lectura:** el mapa describe cobertura de posiciones y puede ocultar puntos superpuestos; las barras distinguen cantidad de sitios de cantidad de viviendas registradas. Un hueco puede reflejar registro o cobertura, no ausencia de turismo. No se interpretan resultados todavía inexistentes.

## 3. Collect Events
Se reúnen únicamente XY idénticas. No hay Integrate, tolerancia de cercanía, deduplicación ni reparación. Se exige igualdad exacta de los conteos por coordenada y suma total. Luego el mapa graduado y el histograma muestran ICOUNT como atributo para Moran.
**Compruebe:** ¿cuenta registros o sitios, y dónde quedan lugares sin registros?


### Collect Events y conservación

Coincidencia XY exacta, sin Integrate. Analogía: contar tarjetas en la misma casilla sin moverlas. Igualdad de conteos por coordenada y suma total son obligatorias.

**Precisión de almacenamiento:** la entrada conserva resolución ≈6.71×10⁻⁹ m y Collect Events crea salida con resolución 0.0001 m. La primera ejecución preservada falló por igualdad binaria de flotantes, aun conservando todos los sitios/conteos. Se comprueba correspondencia uno a uno sin colisiones y diferencia ≤10⁻⁷ m; esto no mueve coordenadas ni introduce radio de agrupación. Un cambio mayor detiene el análisis.

In [ ]:
# Pasar de registros a sitios conservando suma y posición; no deduplicar.
# Cambiar soporte sin desplazar geometrías: de registros a sitios ocupados.
weighted = str(Path(gdb) / 'Viviendas_ICOUNT')
result = arcpy.stats.CollectEvents(working, weighted)
print(result.getMessages())
with arcpy.da.SearchCursor(weighted, ['SHAPE@XY', 'ICOUNT']) as cursor:
    xy_after = {xy: count for xy, count in cursor}
# La GDB de salida cambia resolución; comparar almacenamiento sin modificar geometrías.
# 1e-7 m es límite de comprobación numérica, NO tolerancia de agrupación de eventos.
def numeric_key(xy): return tuple(round(v,7) for v in xy)
before_keys={numeric_key(xy):(xy,count) for xy,count in xy_before.items()}
after_keys={numeric_key(xy):(xy,count) for xy,count in xy_after.items()}
if len(before_keys)!=len(xy_before) or len(after_keys)!=len(xy_after):
    raise ValueError('Colisión en comprobación numérica: no fusionar ni aceptar ambigüedad.')
if set(before_keys)!=set(after_keys) or sum(xy_after.values())!=n:
    raise ValueError('No se conservan sitios y suma total; detener sin ajuste.')
max_storage_delta=0.0
for key,(xy,count) in before_keys.items():
    target,after_count=after_keys[key]
    delta=math.dist(xy,target);max_storage_delta=max(max_storage_delta,delta)
    if count!=after_count or delta>1e-7: raise ValueError('Cambió conteo o posición más allá del límite numérico.')
print('Correspondencia uno a uno; diferencia máxima de almacenamiento (m):',max_storage_delta)
print('Resolución XY entrada/salida (m):',sr.XYResolution,arcpy.Describe(weighted).spatialReference.XYResolution)
print('Registros:', n, 'sitios:', len(xy_after), 'rango ICOUNT:', min(xy_after.values()), max(xy_after.values()))


**Lectura y conclusión:** Menos filas significa cambio de soporte a sitios, no eliminación de viviendas. Una coincidencia no certifica duplicados administrativos.
**Compruebe:** ¿cuenta registros o sitios, y dónde quedan lugares sin registros?


## Describir ICOUNT antes de inferir asociación
Ahora cada fila es un **sitio ocupado**. N es el número de sitios y xᵢ su ICOUNT: COUNT=N, SUM=Σxᵢ recupera los registros originales y media=SUM/N. Analogía: contar tarjetas en casillas ya ocupadas; las vacías no están en la tabla. No conocemos todos los lugares posibles con cero eventos.

MIN/MAX delimitan multiplicidad; mediana es el centro de valores ordenados; moda es el más frecuente (puede haber empates). Q1/Q3 delimitan el 50 % central e IQR=Q3−Q1 mide su amplitud. La desviación muestral s=√[Σ(xᵢ−media)²/(N−1)] expresa dispersión en registros/sitio; la poblacional usa N. Mostrar ambas contrasta convenciones, no demuestra representatividad ni independencia espacial.

ArcPy exporta **solo ICOUNT**, no todos los números, que incluirían IDs. La biblioteca estándar `statistics` permite una comprobación independiente de desviación, media y contabilidad, sin sustituir gráficos o modelos. Los cuartiles se leen de ArcGIS: las convenciones de interpolación pueden diferir.

**Fuentes verificadas por el orquestador, 2026-09-16:** Esri Pro 3.6, [Field Statistics To Table](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/data-management/field-statistics-to-table.htm), Parameters/Python; [Interact with statistics](https://pro.arcgis.com/en/pro-app/3.6/help/analysis/geoprocessing/data-engineering/view-statistics.htm), Select fields/Calculate/Types of statistics. La UI documenta N−1; no demuestra la convención del exportador: se contrasta abajo.

In [ ]:
# Exportar únicamente el atributo que responde a la pregunta, con nombres explícitos.
stat_mapping = [
    ['FIELDNAME', 'Campo'], ['COUNT', 'Cantidad'], ['NULLS', 'Nulos'],
    ['SUM', 'Suma'], ['MINIMUM', 'Minimo'], ['MAXIMUM', 'Maximo'],
    ['MEAN', 'Media'], ['MEDIAN', 'Mediana'], ['MODE', 'Moda'],
    ['STANDARDDEVIATION', 'Desv'], ['FIRSTQUARTILE', 'Q1'],
    ['THIRDQUARTILE', 'Q3'], ['INTERQUARTILERANGE', 'InterquartileRange'],
]
statistics_result = arcpy.management.FieldStatisticsToTable(
    weighted, ['ICOUNT'], gdb, [['NUMERIC', 'Descriptivos_ICOUNT']],
    out_statistics=stat_mapping,
)
print(statistics_result.getMessages())
stats_table = str(Path(gdb) / 'Descriptivos_ICOUNT')
stat_fields = [pair[1] for pair in stat_mapping]
actual_fields = {field.name for field in arcpy.ListFields(stats_table)}
if not set(stat_fields).issubset(actual_fields):
    raise ValueError('La tabla no contiene los nombres solicitados; revisar esquema.')
with arcpy.da.SearchCursor(stats_table, stat_fields) as cursor:
    statistic_rows = list(cursor)
if len(statistic_rows) != 1:
    raise ValueError('Se esperaba una fila estadística para ICOUNT.')
stats = dict(zip(stat_fields, statistic_rows[0]))
# Pro 3.6.2 conserva este nombre físico para IQR; se declara la equivalencia lógica.
stats['IQR'] = stats['InterquartileRange']
print('Tabla ArcGIS, sin selección ni agrupación:', stats)


**Lectura:** una fila resume ICOUNT, no un sitio. Cantidad cuenta valores no nulos; Suma conserva registros. La siguiente celda verifica estas identidades y distingue N de N−1 antes de interpretar dispersión.

**Esquema observado Pro 3.6.2:** el campo de rango intercuartílico se conserva como `InterquartileRange`; `IQR` es solo su nombre lógico en el diccionario Python. No se asume que el exportador siempre acepte abreviaturas.
**Compruebe:** ¿un número justifica promediar una clave, o cero sustituir un ausente?


In [ ]:
# Contrastar fórmulas sin modificar registros; statistics es biblioteca estándar, no un SIG alternativo.
import statistics
with arcpy.da.SearchCursor(weighted, ['ICOUNT']) as cursor:
    icount = [row[0] for row in cursor]
if any(value is None or value < 1 or int(value) != value for value in icount):
    raise ValueError('ICOUNT requiere conteos enteros positivos en sitios ocupados.')
frequency_icount = Counter(icount)
n_sites, n_records = len(icount), sum(icount)
sample_sd = statistics.stdev(icount)
population_sd = statistics.pstdev(icount)
mean_icount = statistics.mean(icount)
checks = {'Cantidad': n_sites, 'Nulos': 0, 'Suma': n_records,
          'Minimo': min(icount), 'Maximo': max(icount),
          'Media': mean_icount, 'Mediana': statistics.median(icount)}
for field, expected in checks.items():
    if not math.isclose(float(stats[field]), expected, rel_tol=1e-10, abs_tol=1e-10):
        raise ValueError(f'Descriptivo no conciliado: {field}.')
match_sample = math.isclose(stats['Desv'], sample_sd, rel_tol=1e-10, abs_tol=1e-12)
match_population = math.isclose(stats['Desv'], population_sd, rel_tol=1e-10, abs_tol=1e-12)
if not (match_sample or match_population):
    raise ValueError('Desviación exportada no coincide con N ni N−1; investigar.')
sd_convention = ('indistinguible entre N y N−1' if match_sample and match_population
                 else 'N−1 (muestral)' if match_sample else 'N (poblacional)')
if not math.isclose(stats['IQR'], stats['Q3'] - stats['Q1'], abs_tol=1e-10):
    raise ValueError('IQR no coincide con Q3−Q1.')
print('Frecuencia exacta ICOUNT → sitios:', dict(sorted(frequency_icount.items())))
print(f'Desviación exportada={stats["Desv"]:.12f}; stdev={sample_sd:.12f}; pstdev={population_sd:.12f}')
print('Convención OBSERVADA de la tabla:', sd_convention)
print('UI documenta N−1: comparar población/selección antes de equiparar resultados.')
print('Moda(s) por frecuencia:', statistics.multimode(icount), '| Moda exportada:', stats['Moda'])


**Contraste histórico — pendiente N04:** desviación ArcGIS=8.246240138568, stdev(N−1)=8.246240138568, pstdev(N)=8.244827506414. Expresan dispersión en registros por sitio; la concordancia con N−1 identifica convención muestral, no independencia espacial ni representatividad. ¿Por qué pueden diferir aunque usen los mismos sitios?

**Lectura del contraste:** la concordancia acredita cálculo sobre el mismo soporte, no calidad administrativa. La convención se decide por las comparaciones numéricas, no por el nombre Desv.

### Centro y extremos sin limpieza
La proporción singleton (ICOUNT=1) muestra cuánto domina el mínimo. Cercas Q1−1.5·IQR y Q3+1.5·IQR son una regla descriptiva: con IQR=0 pueden marcar todos los conteos >1. No se elimina nada. Máximo, su frecuencia y asimetría de momento describen cola, no significancia.

**Antes de leer asimetría:** promedio de cubos de desviaciones dividido por desviación poblacional al cubo; medida adimensional. Positiva indica cola hacia multiplicidades altas, no errores ni prueba de normalidad. Se interpreta con mediana/cuartiles/máximo, sin cambiar registros.

In [ ]:
# Interpretar sitios y cola sin convertir multiplicidad en riesgo ni borrar extremos.
singletons = frequency_icount[1]
lower_fence = stats['Q1'] - 1.5 * stats['IQR']
upper_fence = stats['Q3'] + 1.5 * stats['IQR']
flagged = sum(value < lower_fence or value > upper_fence for value in icount)
# Tercer momento estandarizado descriptivo; no es prueba de normalidad.
skew_moment = (sum((value - mean_icount) ** 3 for value in icount) / n_sites
               / population_sd ** 3) if population_sd else None
print(f'{n_records:,} registros en {n_sites:,} sitios; media={mean_icount:.6f} registros/sitio ocupado.')
print(f'Mediana={stats["Mediana"]}; moda={stats["Moda"]}; mínimo={stats["Minimo"]}; máximo={stats["Maximo"]}.')
print(f'Sitios con un solo registro: {singletons:,}/{n_sites:,} = {singletons/n_sites:.2%}.')
print(f'Q1={stats["Q1"]}; Q3={stats["Q3"]}; IQR={stats["IQR"]}; cercas=[{lower_fence}, {upper_fence}].')
print(f'Fuera de cercas: {flagged:,} sitios ({flagged/n_sites:.2%}); no se borra ninguno.')
if stats['IQR'] == 0:
    print('IQR degenerado: el 50 % central coincide. Marcar otros valores no demuestra errores.')
print(f'Máximo: {max(icount)} registros/sitio en {frequency_icount[max(icount)]} sitios.')
print('Asimetría de momento:', skew_moment, '| cola de multiplicidad, no evidencia de normalidad.')
print('La media no es riesgo/tasa: faltan exposición, sitios con cero y denominador poblacional.')
print('El histograma pierde posición; el mapa muestra dónde están los conteos, no sus causas.')


**Lectura histórica — pendiente N04:** asimetría=11.114977348510617: cola hacia conteos altos, no proporción ni riesgo. Pocos sitios altos elevan la media. ¿Por qué no autoriza eliminarlos?

**Resultado observado:** 7,529 registros en 2,919 sitios. Media=2.579308; mediana=1.0; moda=1.0; mínimo/máximo=1.0/186.0. Q1=1.0, Q3=1.0, IQR=0.0. Singletons=2,295 (78.62%). Desviación exportada=8.246240138568: coincide con N−1 (muestral); stdev(N−1)=8.246240138568, pstdev(N)=8.244827506414. Las cercas marcan 624 sitios, sin eliminar ninguno. IQR=0 concentra el centro en un valor: la regla no identifica errores. La media describe únicamente sitios ocupados, no riesgo/tasa; el mapa aporta posición ausente en el histograma.
**Conclusión:** predominan sitios con una sola fila; la cola eleva la media. Media, mediana, moda y desviación usan registros/sitio; IQR=0 no significa que todos los valores sean iguales. **Compruebe:** ¿por qué 1.5×IQR puede marcar valores legítimos? Cifras históricas, pendiente N04.


### Distribución discreta: tabla completa, barras, caja y mapa
**Entrada:** ICOUNT contrastado. **Operaciones:** frecuencias exactas, panel completo con cola agrupada y panel sin singleton para leer minorías; caja sin estandarización y con puntos fuera de cercas. **Salida:** tabla completa y PNG ArcGIS. Cada intervalo agrupado se etiqueta, no se presenta como valor exacto. La caja muestra mediana/cuartiles/cercas 1.5×IQR; con IQR=0 colapsa en 1 y no justifica borrar observaciones.

El mapa separa 1 y 2 y agrupa la cola; verde claro/oscuro significa menos/más registros, no frío/calor inferencial. Fuente: elaboración docente con datos ejecutados; Esri Pro 3.6 [Bar](https://pro.arcgis.com/en/pro-app/3.6/arcpy/charts/bar.htm) y [Box](https://pro.arcgis.com/en/pro-app/3.6/arcpy/charts/box.htm), Syntax/Parameters/Methods, consulta 2026-09-16.


En las barras, 01 a 10 son conteos exactos; los prefijos 11., 12., 13. y 14. ordenan intervalos de la cola, no son valores adicionales. Los puntos del eje vertical son separadores de miles cuando así los presenta ArcGIS; lea la tabla exacta para evitar ambigüedad.
La etiqueta 101+ agrupa desde 101 hasta el máximo observado (186 en P03), no una categoría abierta con datos desconocidos.

In [ ]:
# Frecuencia exacta: denominador = sitios ocupados, no todo el territorio.
tabla_visible(['Campo', 'Tipo', 'Rol', 'Unidad', 'Ejemplo', 'Nulos / cero'], [
    ['ICOUNT', next(f.type for f in arcpy.ListFields(weighted) if f.name == 'ICOUNT'),
     'Registros originales con XY idéntica', 'registros por sitio', min(icount),
     'No hay nulos ni ceros: sitios sin registros no se incorporan'],
])
tabla_visible(['ICOUNT exacto', 'Sitios', 'Porcentaje de sitios'],
              [[value, count, round(100*count/n_sites, 3)] for value,count in sorted(frequency_icount.items())])
# Valores 1 a 10 exactos; cola agrupada explícitamente, sin truncar el total.
frequency_rows = [(f'{v:02d}', frequency_icount[v]) for v in range(1, min(10, int(max(icount)))+1)]
for lower, upper, label in [(11,20,'11. 11–20'), (21,50,'12. 21–50'), (51,100,'13. 51–100'), (101,int(max(icount)),'14. 101+')]:
    if max(icount) >= lower:
        frequency_rows.append((label, sum(count for value,count in frequency_icount.items() if lower <= value <= upper)))
assert sum(count for _,count in frequency_rows) == n_sites
frequency_table = tabla_frecuencia('Frecuencia_legible', frequency_rows)
chart_png(arcpy.charts.Bar(x='Categoria', y='Sitios', aggregation='SUM', dataSource=frequency_table,
    title='Distribución completa: valores bajos exactos y cola agrupada',
    xTitle='Registros por sitio (intervalos de cola indicados)', yTitle='Número de sitios ocupados'), 'frecuencia_completa')
# Panel sin 1: no altera el universo, permite leer la cola minoritaria.
tail_table = tabla_frecuencia('Frecuencia_cola', frequency_rows[1:])
chart_png(arcpy.charts.Bar(x='Categoria', y='Sitios', aggregation='SUM', dataSource=tail_table,
    title='Detalle de sitios con más de un registro (excluye 1 solo en este panel)',
    xTitle='Registros por sitio (cola agrupada)', yTitle='Número de sitios ocupados'), 'frecuencia_cola')
chart_png(arcpy.charts.Box(y='ICOUNT', standardizeValues=False, showOutliers=True,
    dataSource=weighted, title='ICOUNT: caja en unidades originales y valores fuera de cercas',
    xTitle='Sitios ocupados', yTitle='Registros por sitio'), 'caja_icount')
print(f'Lectura: {frequency_icount[1]:,} sitios tienen un registro; {n_sites-frequency_icount[1]:,} tienen más de uno; máximo={max(icount)}.')
print(f'Q1={stats["Q1"]}; Q3={stats["Q3"]}; IQR={stats["IQR"]}. Caja colapsada en 1: valores mayores no son errores a eliminar.')
# Clasificar únicamente la salida nueva; las clases descriptivas no son significancia.
arcpy.management.AddField(weighted, 'ClaseConteo', 'TEXT', field_length=30)
with arcpy.da.UpdateCursor(weighted, ['ICOUNT','ClaseConteo']) as cursor:
    for row in cursor:
        value = row[0]
        row[1] = '01' if value == 1 else '02' if value == 2 else '03' if value <= 5 else '04' if value <= 10 else '05' if value <= 50 else '06'
        cursor.updateRow(row)
count_styles = {
    '01': ('1 registro', [190,215,190,100], 2),
    '02': ('2 registros', [120,190,130,100], 3),
    '03': ('3–5 registros', [60,160,100,100], 4),
    '04': ('6–10 registros', [20,130,75,100], 5),
    '05': ('11–50 registros', [0,95,50,100], 6),
    '06': ('51–máximo registros', [0,55,30,100], 7),
}
mapa_categorias(weighted, 'ClaseConteo', 'mapa_icount_legible', 'ICOUNT: multiplicidad descriptiva', count_styles,
    'Verde claro a oscuro: menor a mayor conteo, no frío/calor estadístico. Metros del CRS, no precisión terrestre certificada; Web Mercator distorsiona distancias si WKID=3857.')


**Lectura:** use las cifras calculadas bajo los gráficos: el panel completo conserva el denominador; la cola excluye 1 únicamente de la vista. Frecuencia cero en un intervalo significa cero sitios con esa multiplicidad, no territorios sin actividad. La caja colapsada expresa concentración, no una falla de dibujo. No se retiran valores fuera de las cercas.
**Compruebe:** ¿cuenta registros o sitios, y dónde quedan lugares sin registros?


### Data Engineering: ejercicio manual opcional sobre la copia
**Pasos documentados, no ejecutados ni verificados en la interfaz de Pro en esta corrida.** No se inventa `arcpy.DataEngineering`.
1. Abra el APRX de sitios de esta ejecución: apunta a `trabajo.gdb`, no al original. Añada desde esa misma GDB la copia previa a Collect Events, para distinguir registros de sitios.
2. En esa copia abra Data Engineering; añada **LOCALIDAD y SUBCATRNT** con **Add To Statistics** o arrastrándolos; pulse **Calculate**. No agregue IDs/códigos para obtener medias. No active limpieza.
3. Examine frecuencias y previsualización de barras para categorías; contraste cobertura con mapa. Inspeccione Nulls y seleccione los nulos desde su estadística cuando existan. Un cero no es un nulo; no reemplace ni elimine. Anote número seleccionado y total.
4. Quite la selección. En la capa de sitios añada **ICOUNT** y calcule: COUNT=sitios, SUM=registros; centro/dispersión describen multiplicidad. Abra el histograma numérico y relaciónelo con el mapa graduado.
5. Seleccione temporalmente ICOUNT > 1 y recalcule: cambia el denominador. Esas estadísticas no deben coincidir con toda la capa. Quite la selección y calcule otra vez.
6. Abra `Descriptivos_ICOUNT` y compare campo, población, nulos, COUNT, SUM, media y cuartiles. La UI documenta desviación N−1: consulte la convención **observada** del exportador (puede ser N) antes de atribuir diferencias a datos sucios.
7. Documente su observación manual aparte; no guarde cambios sobre originales, no repare, deduplique ni impute. Este itinerario no equivale a interacción ejecutada.

Fuente: Esri, Pro 3.6, [Interact with statistics](https://pro.arcgis.com/en/pro-app/3.6/help/analysis/geoprocessing/data-engineering/view-statistics.htm), selección/cálculo/tipos; pasaje verificado por el orquestador el 2026-09-16.

### Histograma del soporte ocupado
**Entrada:** ICOUNT. **Operación:** histograma ArcGIS, hasta 20 intervalos (4/10 para rangos pequeños). **Salida:** sitios por intervalo; eje x = registros por sitio, eje y = cantidad de sitios. Es una vista global continua: no interpretar límites decimales como conteos posibles. Para valores exactos bajos y cola consulte las barras/tabla anteriores. El mapa descriptivo anterior separa 1 y 2, sin atribuir significancia.

In [ ]:
# Relacionar distribución de multiplicidad con ubicación, sin inferir riesgo.
chart_png(arcpy.charts.Histogram(x='ICOUNT', dataSource=weighted, binCount=min(20, int(max(icount))), xTitle='ICOUNT: registros por sitio', yTitle='Número de sitios ocupados', title='Viviendas registradas por sitio ocupado'), 'histograma_icount')
if len(set(xy_after.values())) < 2:
    raise ValueError('ICOUNT constante: este Moran no está definido, aunque la nube parezca agrupada.')

**Lectura y conclusión:** No se incluyen lugares sin registros como ceros. ICOUNT no es precio, disponibilidad, tasa ni cobertura del mercado.


La revisión anterior corresponde a la corrida histórica; evaluar nuevos paneles después de ejecutar.

**Lectura:** suma conservada no certifica que cada registro represente una vivienda única. ICOUNT no es precio, disponibilidad, ocupación o tasa. Los sitios sin registros no aparecen como ceros, por lo que este contraste no prueba aleatoriedad espacial completa de toda la ciudad.

## 4. Moran global con informe HTML
INVERSE_DISTANCE da mayor influencia a vecinos cercanos; EUCLIDEAN_DISTANCE usa distancia recta y ROW estandariza por filas. **Umbral vacío** calcula un límite que asegura al menos un vecino para cada entidad; **cero** significa sin umbral. No se selecciona por optimizar p ni se fija una distancia histórica como si fuera parámetro nuevo.

El resultado proporciona valores y ruta de informe; se consultan nombres reales de salidas y mensajes, sin asumir a ciegas un índice de ejemplo. El scratch configurado mantiene el informe en la ejecución. Si se produce fuera, se detiene para revisar aislamiento antes de continuar.
**Compruebe:** ¿cuenta registros o sitios, y dónde quedan lugares sin registros?


### Leer Moran sin convertirlo en un mapa de riesgo
**Pregunta:** ¿se parecen los valores **ICOUNT** entre sitios ocupados vecinos? No contrasta directamente densidad de puntos originales ni identifica por sí solo un hotspot local.

| Columna | Lectura sencilla |
|---|---|
| Distance | Distancia efectiva en metros del CRS; modifica vecinos. En incremental son radios acumulados, no anillos. |
| MoransI / I | Asociación adimensional entre desviaciones de ICOUNT de vecinos. Comparar con esperado y pesos: no tiene límites universales −1/+1 ni expresa porcentaje de riesgo. |
| ExpectedI | Esperado bajo H₀ y el soporte/pesos empleados; leer cada fila, sin imponer el número original de sitios a todas las bandas. |
| Variance | Variabilidad de **I bajo H₀**, no varianza de ICOUNT, coordenadas o distancias. Se usa para estandarizar. |
| z_score / ZScore | (I−ExpectedI)/√Variance: cuántas desviaciones nulas separan observado y esperado. Signo=encima/debajo del esperado, no bueno/malo. |
| p_value / PValue | Compatibilidad de un resultado al menos tan extremo con H₀; no probabilidad de que H₀ sea cierta. 0 mostrado significa por debajo de la precisión de representación/cálculo, no probabilidad matemáticamente nula. |

I por encima del esperado favorece semejanza vecinal; por debajo, contraste, condicionado a pesos y soporte. Un z grande y p pequeño puede acompañar I pequeño: significancia no es magnitud ni causalidad. El 95 % es referencia **nominal de un contraste fijado**, no corrección después de mirar curvas. Primer/máximo pico local de z y máximo numérico no son lo mismo. Conservar todas las bandas; seleccionar pico o segunda exploración con los mismos datos no es validación independiente. Referencia: pasajes Esri Pro 3.6 sobre autocorrelación ya citados en este notebook, sin nueva consulta web. **Compruebe:** ¿qué columna habla de magnitud y cuál de separación estandarizada?


In [ ]:
# Firma comprobada en ayuda local ArcPy Pro 3.6.2; umbral None conserva cálculo automático.
result = arcpy.stats.SpatialAutocorrelation(weighted, 'ICOUNT', 'GENERATE_REPORT', 'INVERSE_DISTANCE', 'EUCLIDEAN_DISTANCE', 'ROW', None)
print(result.getMessages())
outputs = {}
output_names = [p.name for p in arcpy.GetParameterInfo('SpatialAutocorrelation_stats') if p.direction == 'Output']
if len(output_names) != result.outputCount:
    raise RuntimeError('Cantidad de salidas distinta al contrato instalado; revisar antes de interpretar.')
for name in output_names:
    value = result.getOutput(name)
    outputs[name] = str(value)
    print(name, value)
html_paths = [Path(value) for value in outputs.values() if str(value).lower().endswith(('.html', '.htm'))]
if not html_paths:
    raise RuntimeError('No se identificó el informe HTML; inspeccionar salidas reales antes de seguir.')
for report in html_paths:
    if not report.is_file() or not report.resolve().is_relative_to(run.resolve()):
        raise RuntimeError('El informe no está disponible dentro de la ejecución aislada; revisar scratch.')
print('Abrir el HTML local indicado para leer I, esperado, z, p y distancia calculada; no es publicación.')


**Lectura histórica — pendiente N04:** Global: I=0.025782 frente a esperado −0.000343, varianza nula mostrada 0.000006, z=10.661289 y p mostrado 0. La varianza redondeada no permite reconstruir exactamente z. Umbral automático=4 822.1390 m; advertencias 000853/001420/001422 informan umbral y entidades con más de 1 000 vecinos. Asociación de ICOUNT bajo estos pesos, no efecto grande ni hotspot de localizaciones. ¿Por qué p mostrado 0 no es certeza?

**Interpretación del informe:** z = (I − E[I]) / raíz de la varianza nula. La varianza corresponde a I bajo H₀, no a distancias entre vecinos. p es extremidad bajo el modelo, no P(H₀). Un z grande no mide importancia económica ni causalidad; no rechazo no demuestra azar. El umbral real debe copiarse del informe/mensajes observados, nunca de una cifra histórica.

## 5. Dos exploraciones de escala
Configuraciones confirmadas por el orquestador: 20/2000/200 y 30/1000/200. Se calculan y se interpretan por separado; no se agrega clustering.

**Esta ejecución:** umbral automático 4 822.1390 m, I=0.025782, z=10.661289 y p mostrado/exportado como 0 por la herramienta: no interpretarlo como probabilidad exactamente nula. Asociación positiva de magnitud pequeña respecto al soporte, con evidencia estandarizada fuerte bajo los pesos especificados; no causalidad ni magnitud económica. Avisos 000853 (umbral calculado), 001420/001422 (entidades con más de 1 000 vecinos) se conservan; no son errores de ejecución. El contrato instalado denomina la salida del índice `Index`, aunque la documentación la describe como Moran’s I: se leen los nombres reales, no un índice fijo.

### Calcular incremental Inicial

ICOUNT; 20 bandas; inicio 2000 m, paso 200 m; final solicitado 5800 m. EUCLIDEAN / ROW_STANDARDIZATION. Radios acumulativos, no anillos. Diseño distinto de la distancia inversa global; no confirmación independiente. Registrar distancias reales y posibles ajustes.

In [ ]:
# Evaluar bandas previstas y conservar distancias reales y mensajes completos.
name='Inicial';bands=20;beginning=2000;step=200
table = str(Path(gdb) / ('Incremental_' + name))
report = str(run / ('incremental_' + name + '.pdf'))
result = arcpy.stats.IncrementalSpatialAutocorrelation(weighted, 'ICOUNT', bands, beginning, step, 'EUCLIDEAN', 'ROW_STANDARDIZATION', table, report)
print(name, 'solicitado:', bands, beginning, step, result.getMessages())
columns = ['Distance', 'MoransI', 'ExpectedI', 'Variance', 'z_score', 'p_value']
if not set(columns).issubset({f.name for f in arcpy.ListFields(table)}):
    raise ValueError('Esquema incremental inesperado; revisar sin inventar campos.')
with arcpy.da.SearchCursor(table, columns) as cursor:
    rows = sorted(list(cursor))
print(columns)
for row in rows:
    print(row)


**Primera fila histórica — pendiente N04:** Distance=2000.00 m, I=0.015649, ExpectedI=-0.000343, Variance=0.000006, z=6.280796, p=0.000000. La distancia fija vecinos; I se compara con ese esperado. Varianza es incertidumbre nula de I: z expresa separación en desviaciones nulas, no registros ni metros. El redondeo puede impedir reconstruir z exactamente. Conclusión: interpretar esta fila junto a todas las bandas, sin elegir solo el menor p. Mensajes conservados: First Peak (Distance; Value): 2200.00; 7.026597; Max Peak (Distance; Value): 5000.00; 10.637299. **Compruebe:** ¿esa p demuestra que H₀ tiene esa probabilidad? No.

**Lectura y conclusión:** I resume productos de desviaciones entre vecinos. z=(I−E[I])/√Var₀(I), varianza del estadístico bajo H₀, no de distancias. p no es P(H₀); conservar todas las bandas y mensajes.
**Compruebe:** ¿qué cambia al modificar vecinos y por qué repetir datos no valida independientemente?


### Inicial — I observado

Eje x: distancia efectiva; y: I observado. Leer las tres curvas con la tabla, no solo el resultado más llamativo.

In [ ]:
# Representar la medida indicada frente a distancia real, no seleccionar solo el menor p.
field='MoransI'
chart_png(arcpy.charts.Line(x='Distance', y=field, dataSource=table, title=name + ': ' + {'MoransI':'Moran I observado', 'z_score':'Separación estandarizada respecto al azar', 'p_value':'Valor p nominal'}[field], xTitle='Distancia real de tabla (m)', yTitle={'MoransI':'I de Moran (adimensional)', 'z_score':'z (desviaciones estándar bajo H₀)', 'p_value':'p nominal (no P de H₀)'}[field]), name + '_' + field)


**Lectura y conclusión:** Pico local y máximo numérico pueden diferir. Los p son nominales y dependientes entre bandas. Un pico es candidato según pregunta, no escala óptima ni causa.

**Valores de esta curva (Inicial):** 20 distancias, 2000–5800 m; MoransI mínimo=0.0077572728 a 5800 m y máximo=0.015649197 a 2000 m. La magnitud del índice no expresa probabilidad ni número de registros.
**Cifras históricas — pendiente N04. Compruebe:** ¿Cambió asociación o solo significancia? Compare I con ExpectedI, no con un porcentaje de riesgo.


### Inicial — z estandarizado

Eje x: distancia efectiva; y: z estandarizado. Leer las tres curvas con la tabla, no solo el resultado más llamativo.

In [ ]:
# Representar la medida indicada frente a distancia real, no seleccionar solo el menor p.
field='z_score'
chart_png(arcpy.charts.Line(x='Distance', y=field, dataSource=table, title=name + ': ' + {'MoransI':'Moran I observado', 'z_score':'Separación estandarizada respecto al azar', 'p_value':'Valor p nominal'}[field], xTitle='Distancia real de tabla (m)', yTitle={'MoransI':'I de Moran (adimensional)', 'z_score':'z (desviaciones estándar bajo H₀)', 'p_value':'p nominal (no P de H₀)'}[field]), name + '_' + field)


**Lectura y conclusión:** Pico local y máximo numérico pueden diferir. Los p son nominales y dependientes entre bandas. Un pico es candidato según pregunta, no escala óptima ni causa.

**Valores de esta curva (Inicial):** 20 distancias, 2000–5800 m; z_score mínimo=6.2807959 a 2000 m y máximo=10.637299 a 5000 m. El máximo numérico no es necesariamente pico local interior; use los mensajes de la herramienta.
**Cifras históricas — pendiente N04. Compruebe:** ¿Un z mayor exige I mayor? Revise varianza nula; el pico no es escala óptima certificada.


### Inicial — p nominal

Eje x: distancia efectiva; y: p nominal. Leer las tres curvas con la tabla, no solo el resultado más llamativo.

In [ ]:
# Representar la medida indicada frente a distancia real, no seleccionar solo el menor p.
field='p_value'
chart_png(arcpy.charts.Line(x='Distance', y=field, dataSource=table, title=name + ': ' + {'MoransI':'Moran I observado', 'z_score':'Separación estandarizada respecto al azar', 'p_value':'Valor p nominal'}[field], xTitle='Distancia real de tabla (m)', yTitle={'MoransI':'I de Moran (adimensional)', 'z_score':'z (desviaciones estándar bajo H₀)', 'p_value':'p nominal (no P de H₀)'}[field]), name + '_' + field)


**Lectura y conclusión:** Pico local y máximo numérico pueden diferir. Los p son nominales y dependientes entre bandas. Un pico es candidato según pregunta, no escala óptima ni causa.

**Valores de esta curva (Inicial):** 20 distancias, 2000–5800 m; p_value mínimo=0 a 2200 m y máximo=3.3684404e-10 a 2000 m. Valores pequeños son p nominales, no probabilidad de H₀; las bandas no son independientes. Un 0 mostrado refleja precisión numérica, no imposibilidad lógica.
**Cifras históricas — pendiente N04. Compruebe:** ¿p pequeño demuestra causa? No: sigue siendo nominal tras explorar bandas.


### Calcular incremental Segunda

ICOUNT; 30 bandas; inicio 1000 m, paso 200 m; final solicitado 6800 m. EUCLIDEAN / ROW_STANDARDIZATION. Radios acumulativos, no anillos. Diseño distinto de la distancia inversa global; no confirmación independiente. Registrar distancias reales y posibles ajustes.

In [ ]:
# Evaluar bandas previstas y conservar distancias reales y mensajes completos.
name='Segunda';bands=30;beginning=1000;step=200
table = str(Path(gdb) / ('Incremental_' + name))
report = str(run / ('incremental_' + name + '.pdf'))
result = arcpy.stats.IncrementalSpatialAutocorrelation(weighted, 'ICOUNT', bands, beginning, step, 'EUCLIDEAN', 'ROW_STANDARDIZATION', table, report)
print(name, 'solicitado:', bands, beginning, step, result.getMessages())
columns = ['Distance', 'MoransI', 'ExpectedI', 'Variance', 'z_score', 'p_value']
if not set(columns).issubset({f.name for f in arcpy.ListFields(table)}):
    raise ValueError('Esquema incremental inesperado; revisar sin inventar campos.')
with arcpy.da.SearchCursor(table, columns) as cursor:
    rows = sorted(list(cursor))
print(columns)
for row in rows:
    print(row)


**Primera fila histórica — pendiente N04:** Distance=1000.00 m, I=0.020779, ExpectedI=-0.000344, Variance=0.000021, z=4.570854, p=0.000005. La distancia fija vecinos; I se compara con ese esperado. Varianza es incertidumbre nula de I: z expresa separación en desviaciones nulas, no registros ni metros. El redondeo puede impedir reconstruir z exactamente. Conclusión: interpretar esta fila junto a todas las bandas, sin elegir solo el menor p. Mensajes conservados: First Peak (Distance; Value): 1400.00; 5.447021; Max Peak (Distance; Value): 5000.00; 10.637299. **Compruebe:** ¿esa p demuestra que H₀ tiene esa probabilidad? No.

**Lectura y conclusión:** I resume productos de desviaciones entre vecinos. z=(I−E[I])/√Var₀(I), varianza del estadístico bajo H₀, no de distancias. p no es P(H₀); conservar todas las bandas y mensajes.
**Compruebe:** ¿qué cambia al modificar vecinos y por qué repetir datos no valida independientemente?


### Segunda — I observado

Eje x: distancia efectiva; y: I observado. Leer las tres curvas con la tabla, no solo el resultado más llamativo.

In [ ]:
# Representar la medida indicada frente a distancia real, no seleccionar solo el menor p.
field='MoransI'
chart_png(arcpy.charts.Line(x='Distance', y=field, dataSource=table, title=name + ': ' + {'MoransI':'Moran I observado', 'z_score':'Separación estandarizada respecto al azar', 'p_value':'Valor p nominal'}[field], xTitle='Distancia real de tabla (m)', yTitle={'MoransI':'I de Moran (adimensional)', 'z_score':'z (desviaciones estándar bajo H₀)', 'p_value':'p nominal (no P de H₀)'}[field]), name + '_' + field)


**Lectura y conclusión:** Pico local y máximo numérico pueden diferir. Los p son nominales y dependientes entre bandas. Un pico es candidato según pregunta, no escala óptima ni causa.

**Valores de esta curva (Segunda):** 30 distancias, 1000–6800 m; MoransI mínimo=0.0056156394 a 6800 m y máximo=0.020779413 a 1000 m. La magnitud del índice no expresa probabilidad ni número de registros.
**Cifras históricas — pendiente N04. Compruebe:** ¿Cambió asociación o solo significancia? Compare I con ExpectedI, no con un porcentaje de riesgo.


### Segunda — z estandarizado

Eje x: distancia efectiva; y: z estandarizado. Leer las tres curvas con la tabla, no solo el resultado más llamativo.

In [ ]:
# Representar la medida indicada frente a distancia real, no seleccionar solo el menor p.
field='z_score'
chart_png(arcpy.charts.Line(x='Distance', y=field, dataSource=table, title=name + ': ' + {'MoransI':'Moran I observado', 'z_score':'Separación estandarizada respecto al azar', 'p_value':'Valor p nominal'}[field], xTitle='Distancia real de tabla (m)', yTitle={'MoransI':'I de Moran (adimensional)', 'z_score':'z (desviaciones estándar bajo H₀)', 'p_value':'p nominal (no P de H₀)'}[field]), name + '_' + field)


**Lectura y conclusión:** Pico local y máximo numérico pueden diferir. Los p son nominales y dependientes entre bandas. Un pico es candidato según pregunta, no escala óptima ni causa.

**Valores de esta curva (Segunda):** 30 distancias, 1000–6800 m; z_score mínimo=4.5708539 a 1000 m y máximo=10.637299 a 5000 m. El máximo numérico no es necesariamente pico local interior; use los mensajes de la herramienta.
**Cifras históricas — pendiente N04. Compruebe:** ¿Un z mayor exige I mayor? Revise varianza nula; el pico no es escala óptima certificada.


### Segunda — p nominal

Eje x: distancia efectiva; y: p nominal. Leer las tres curvas con la tabla, no solo el resultado más llamativo.

In [ ]:
# Representar la medida indicada frente a distancia real, no seleccionar solo el menor p.
field='p_value'
chart_png(arcpy.charts.Line(x='Distance', y=field, dataSource=table, title=name + ': ' + {'MoransI':'Moran I observado', 'z_score':'Separación estandarizada respecto al azar', 'p_value':'Valor p nominal'}[field], xTitle='Distancia real de tabla (m)', yTitle={'MoransI':'I de Moran (adimensional)', 'z_score':'z (desviaciones estándar bajo H₀)', 'p_value':'p nominal (no P de H₀)'}[field]), name + '_' + field)


**Lectura y conclusión:** Pico local y máximo numérico pueden diferir. Los p son nominales y dependientes entre bandas. Un pico es candidato según pregunta, no escala óptima ni causa.

**Valores de esta curva (Segunda):** 30 distancias, 1000–6800 m; p_value mínimo=0 a 2200 m y máximo=4.8574077e-06 a 1000 m. Valores pequeños son p nominales, no probabilidad de H₀; las bandas no son independientes. Un 0 mostrado refleja precisión numérica, no imposibilidad lógica.
**Cifras históricas — pendiente N04. Compruebe:** ¿p pequeño demuestra causa? No: sigue siendo nominal tras explorar bandas.


## Lectura comparada y evaluación
Las distancias producidas pueden diferir de las solicitadas si la herramienta ajusta el rango. Lea mensajes y tabla completa; no elija solo el máximo de z. Picos significativos son candidatos dependientes de la pregunta; no tener pico también es un resultado. Varias bandas comparten datos y vecinos: sus p nominales no incorporan selección de escala ni son pruebas independientes.

El global usa distancia inversa; el incremental explora bandas acumulativas. No atribuir diferencias únicamente al radio ignorando la regla de pesos. No transferir automáticamente una escala a OPTICS ni inferir precio, ocupación o causalidad desde ICOUNT.

**Entregable:** procedencia verificada, diccionario temático cuando esté disponible, calidad, mapa y gráficos leídos, parámetros y distancia global real, tablas completas y una conclusión con límites. Entrada y configuraciones verificadas; la ejecución actual se interpreta debajo.

## Fuentes y estado documental
Datos requeridos: viviendas turísticas Bogotá, Catastro/IDECA, registro administrativo; copia mínima local autorizada disponible. Solo se consultó el endpoint público autorizado al preparar la copia; el notebook funciona offline.
Esri, ArcGIS Pro 3.6, pasajes consultados 2026-09-15: [Collect Events](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/collect-events.htm), Usage/Python; [Incremental Spatial Autocorrelation](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/incremental-spatial-autocorrelation.htm), Usage/Parameters; [cómo funciona incremental](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/how-incremental-spatial-autocorrelation-works.htm), escala/picos; [cómo funciona Moran global](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/h-how-spatial-autocorrelation-moran-s-i-spatial-st.htm), interpretación. Firma y umbral de global: ayuda instalada `arcpy.stats.SpatialAutocorrelation.__doc__`, Pro 3.6.2, comprobada 2026-09-15; cotejo oficial completado por el orquestador el 2026-09-16: Esri, Pro 3.6, [Spatial Autocorrelation](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/spatial-autocorrelation.htm), Usage/Parameters/Python. Umbral euclidiano vacío garantiza al menos un vecino por entidad; cero, con distancia inversa, significa sin umbral. Salidas: MoransI, ZScore, PValue y Report_File.


## Ejecución histórica anterior a esta revisión docente
Inicio UTC: 2026-09-16T17:15:25.013142+00:00; duración de ejecución de celdas 108.017 s; 21 celdas; 11 PNG incrustados; errores: ninguno.

Salidas actuales: `99 - Recursos/salidas_clase_02/practica_03/ejecucion_5bd614af4469`. Proceso nuevo del intérprete Pro; scratch/temp aislados. Los mensajes y advertencias se conservan; duración computacional, no agenda ni ensayo. Data Engineering UI documentada, no interacción observada. No acredita Obsidian ni clase completa.

**Comprobación focal:** PNG de histograma y mapa de sitios abiertos e inspeccionados; títulos, ejes, leyendas y límites se interpretan junto a cada figura. JSON/nbformat, AST, rutas iniciales, comentarios, explicación antes/después, contadores consecutivos, ausencia de errores guardados e igualdad de PNG incrustados con archivos de esta corrida comprobados. Descriptivos releídos de la GDB y contrastados nuevamente con `statistics`; las tres tablas usan N−1. No se ejecutó la interfaz Data Engineering ni se comprobó Obsidian.

**Fuente de esta ampliación:** complemento docente sobre el orden existente, no ejercicio nuevo. El orquestador revisó en esta sesión grabación 14 (2026-06-24), segmentos focales 65:35–65:48, 66:20–66:34, 69:30–69:52 y 70:20–70:40 (Data Engineering, categorías, Collect Events/ICOUNT). Cobertura parcial; no se afirma visionado completo.

## Complemento solicitado: Getis-Ord Gi* (no mostrado en la grabación)
**Pregunta:** ¿hay sitios con multiplicidad alta o baja rodeados de multiplicidades semejantes? Un valor alto aislado no basta para un punto caliente. Gi* produce un contraste por sitio; no es EDA universal.

**Diseño previo:** ICOUNT de sitios ocupados; ocho vecinos más cercanos, distancia euclidiana y FDR. Esri recomienda aproximadamente ocho vecinos con asimetría: k=8 se fija antes del resultado, no se busca para producir colores. La distancia física varía; ningún pico incremental se usa como radio. `NONE` es parámetro heredado sin efecto.

**Supuestos:** al menos 30 sitios y variación finita; CRS/extensión inspeccionados arriba. Se preserva el CRS fuente. Con WKID=3857, especialmente en cobertura nacional, Web Mercator distorsiona distancias y vecindades: análisis exploratorio, no precisión terrestre en metros. FDR controla pruebas locales dentro de esta corrida, no selección de k/bandas entre modelos.

**Entrada → operación → salida:** sitios/ICOUNT → una llamada HotSpots → copia Gi_8vecinos_FDR con z, p nominal y categoría FDR. Se conservan mensajes y sitios; sin Integrate, reparación, imputación ni eliminación.

**Fuente complementaria:** Esri, ArcGIS Pro 3.6, [Hot Spot Analysis: Usage/Parameters/Python](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/hot-spot-analysis.htm) y [How Hot Spot Analysis works](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/h-how-hot-spot-analysis-getis-ord-gi-spatial-stati.htm), consulta 2026-09-16. Firma contrastada con Pro 3.6.2 instalado.

**Lectura de Gi\*:** combina valor del sitio y vecinos; k=8 fija vecinos, no radio físico. GiZScore es separación estandarizada (positivo: concentración alta; negativo: baja); GiPValue es p crudo, no probabilidad de riesgo. Gi_Bin usa FDR: ±1/±2/±3 son categorías 90/95/99 %, con signo caliente/frío; 0=no significativo, no prueba de azar ni lugar seguro. FDR ajusta múltiples contrastes locales, no selección de modelo, k o escala después de ver resultados. **Compruebe:** ¿por qué GiPValue y Gi_Bin no comparten interpretación cruda?


In [ ]:
# Configuración fijada antes de observar resultados; entradas de solo lectura.
import inspect, json
print('Firma instalada:', inspect.signature(arcpy.stats.HotSpots))
assert 'number_of_neighbors' in inspect.signature(arcpy.stats.HotSpots).parameters
if n_sites < 30 or n_sites <= 8 or len(set(icount)) < 2 or not all(math.isfinite(v) for v in icount):
    raise ValueError('Gi* requiere al menos 30 sitios, más de ocho, valores finitos y variación.')
if sr.type != 'Projected' or sr.linearUnitName.lower() not in ('meter','metre'):
    raise ValueError('Revisar CRS y extensión: no proyectar automáticamente.')
hotspots = str(Path(gdb) / 'Gi_8vecinos_FDR')
gi_result = arcpy.stats.HotSpots(
    Input_Feature_Class=weighted, Input_Field='ICOUNT', Output_Feature_Class=hotspots,
    Conceptualization_of_Spatial_Relationships='K_NEAREST_NEIGHBORS',
    Distance_Method='EUCLIDEAN_DISTANCE', Standardization='NONE',
    Apply_False_Discovery_Rate__FDR__Correction='APPLY_FDR', number_of_neighbors=8,
)
print(gi_result.getMessages())
(run / 'Gi_mensajes.txt').write_text(gi_result.getMessages(), encoding='utf-8')
with arcpy.da.SearchCursor(hotspots, ['Gi_Bin']) as cursor:
    gi_counts = Counter(int(row[0]) for row in cursor)
assert sum(gi_counts.values()) == n_sites
labels = {-3:'Frío 99%', -2:'Frío 95%', -1:'Frío 90%', 0:'No significativo', 1:'Caliente 90%', 2:'Caliente 95%', 3:'Caliente 99%'}
tabla_visible(['Gi_Bin (FDR)', 'Categoría', 'Sitios'], [[v, labels[v], gi_counts[v]] for v in range(-3,4)])
tabla_visible(['Campo real', 'Significado', 'Lectura y límite'], [
    ['GiZScore', 'Estadístico local estandarizado', 'Positivo/negativo: valores altos/bajos agrupados, no riesgo'],
    ['GiPValue', 'p nominal sin corrección FDR', 'Extremidad bajo el modelo nulo; no probabilidad de H0'],
    ['Gi_Bin', 'Clasificación FDR: ±1/±2/±3 y 0', 'Frío/caliente; 90/95/99%; 0 no significativo'],
    ['SOURCE_ID', 'Enlace al identificador local de entrada', 'No clave semántica ni prueba de duplicación'],
    ['ICOUNT', 'Registros por sitio ocupado', 'No ausencia territorial, exposición ni tasa'],
])
(run / 'Gi_conteos.json').write_text(json.dumps({str(v):gi_counts[v] for v in range(-3,4)}, indent=2), encoding='utf-8')


### Leer categorías y después el mapa
**Resultado:** siete categorías incluso vacías; mensajes preservan advertencias. GiZScore/GiPValue son crudos; Gi_Bin refleja FDR. SOURCE_ID enlaza fila local, no entidad certificada.

**Siguiente operación:** capa Gi* → azul/gris/rojo y barras → PNG estáticos. Porcentajes de leyenda son niveles de significancia, no porcentajes de sitios ni probabilidades de causalidad.
Barras: NS = no significativo; Cal = caliente; Frío = agrupación baja. Los prefijos 1–7 solo ordenan las siete categorías; 90/95/99% son niveles, no proporciones de sitios.

In [ ]:
# Azul-neutral-rojo igual en las tres prácticas; no fabricar categorías presentes.
gi_colors = [[33,102,172,100], [67,147,195,100], [146,197,222,100], [190,190,190,100],
             [244,165,130,100], [214,96,77,100], [178,24,43,100]]
arcpy.management.AddField(hotspots, 'ClaseGi', 'TEXT', field_length=40)
with arcpy.da.UpdateCursor(hotspots, ['Gi_Bin','ClaseGi']) as cursor:
    for row in cursor:
        row[1] = str(int(row[0])+4) + '. ' + labels[int(row[0])]
        cursor.updateRow(row)
gi_styles = {str(v+4)+'. '+labels[v]:(labels[v], gi_colors[v+3], 3 if v == 0 else 5) for v in range(-3,4)}
mapa_categorias(hotspots, 'ClaseGi', 'mapa_Gi_FDR', 'Gi*: ocho vecinos, clasificación FDR', gi_styles,
    'Azul=frío; gris=no significativo; rojo=caliente. Sitios ocupados, no riesgo.' + chr(10) +
    'Vecinos fijados antes del resultado; alcance físico variable; categorías ausentes no tienen puntos.')
short_labels = ['1 Frío99%', '2 Frío95%', '3 Frío90%', '4 NS', '5 Cal90%', '6 Cal95%', '7 Cal99%']
gi_table = tabla_frecuencia('Frecuencia_Gi', [(short_labels[v+3],gi_counts[v]) for v in range(-3,4)])
chart_png(arcpy.charts.Bar(x='Categoria', y='Sitios', aggregation='SUM', dataSource=gi_table,
    title='Gi*: sitios por categoría FDR (incluye categorías con cero sitios)',
    xTitle='Categoría inferencial de frío a caliente', yTitle='Número de sitios ocupados'), 'barras_Gi_FDR')
cold = sum(gi_counts[v] for v in (-3,-2,-1))
hot = sum(gi_counts[v] for v in (1,2,3))
print(f'Resultado observado: {cold} fríos, {gi_counts[0]} no significativos y {hot} calientes. Total={n_sites}.')
print('Categorías vacías son válidas: no se modifica k para obtener colores. Multiplicidad local no equivale a ausencia o riesgo.')

# Detalle explícito: seis categorías significativas; NS permanece en el panel completo.
gi_detalle = tabla_frecuencia('Frecuencia_Gi_detalle',
    [(short_labels[v+3], gi_counts[v]) for v in (-3,-2,-1,1,2,3)])
chart_png(arcpy.charts.Bar(x='Categoria', y='Sitios', aggregation='SUM', dataSource=gi_detalle,
    title='Detalle Gi*: categorías significativas (NS excluido solo aquí)',
    xTitle='Categoría FDR; ceros conservados', yTitle='Sitios ocupados'), 'barras_Gi_detalle')
tabla_visible(['Categoría FDR', 'Sitios', 'Total de sitios', 'Porcentaje del total'],
    [[labels[v], gi_counts[v], n_sites, round(100*gi_counts[v]/n_sites, 3)] for v in range(-3,4)])
print('Lectura: detalle sin NS para comparar minorías; cero fríos no significa un territorio más seguro.')


**Lectura histórica — pendiente N04:** Gi_Bin en orden −3,−2,−1,0,+1,+2,+3: **0/0/0/2802/18/18/81** sitios. Barras=conteos de sitios ocupados; mapa=categorías locales, no probabilidad de riesgo. No hubo fríos significativos: es válido, sin cambiar k para fabricarlos. Hay categorías calientes bajo este diseño, sin demostrar causas ni seguridad de sitios neutros. ¿El denominador son sitios o registros?

**Conclusión:** lea el recuento ejecutado bajo las barras. Frío indica agrupación local de multiplicidades bajas respecto al conjunto ocupado; no ausencia geográfica, bajo riesgo ni menor exposición. Caliente tampoco implica causalidad. No significación no demuestra ausencia de estructura. FDR no corrige exploración entre modelos. Complemento no atribuido a la grabación; la cobertura de video continúa parcial.

**Observación de esta corrida:** 0 sitios fríos; 2802 no significativos; calientes 90%/95%/99%: 18/18/81. No hay azul porque ninguna categoría fría quedó significativa tras FDR; no se altera k. Los mapas y barras son consistentes con esta tabla, no con una expectativa de tener ambos colores.

## Ejecución previa: antes de ajustar etiquetas truncadas
Proceso limpio ArcGIS Pro; 25 celdas; 48.832 s; 15 PNG incrustados; cero errores. Salidas: `99 - Recursos/salidas_clase_02/practica_03/ejecucion_da2dea95575c`. Duración de cómputo, no agenda ni ensayo. Cierre explícito tras guardar por fallo nativo de finalización observado en P01. Aprobación académica pendiente.

## Ejecución histórica anterior al atlas
Proceso limpio ArcGIS Pro; 25 celdas; 46.345 s; 15 PNG incrustados; cero errores. Salidas: `99 - Recursos/salidas_clase_02/practica_03/ejecucion_235fceae8af2`. Duración de cómputo, no agenda ni ensayo. Cierre explícito tras guardar por fallo nativo de finalización observado en P01. Aprobación académica pendiente.